In [281]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CustomerApp") \
    .getOrCreate()
customers = [

(1, "Rahul", "Hyderabad"),
(2, "Priya", "Bangalore"),
(3, "Amit", "Mumbai"),
(4, "Sneha", "Chennai"),
(5, "Farhan", "Delhi")

]

customer_columns = [
    "customer_id",
    "customer_name",
    "city"
]

customers_df = spark.createDataFrame(
    customers,
    customer_columns
)

customers_df.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|          1|        Rahul|Hyderabad|
|          2|        Priya|Bangalore|
|          3|         Amit|   Mumbai|
|          4|        Sneha|  Chennai|
|          5|       Farhan|    Delhi|
+-----------+-------------+---------+



In [282]:
orders = [

(101, 1, "Laptop", 65000),
(102, 2, "Mobile", 25000),
(103, 1, "TV", 45000),
(104, 3, "Chair", 5000),
(105, 7, "Watch", 8000)

]

order_columns = [
    "order_id",
    "customer_id",
    "product",
    "amount"
]

orders_df = spark.createDataFrame(
    orders,
    order_columns
)

In [283]:
customers_df.join(orders_df,"customer_id","inner").show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [284]:
customers_df.join(orders_df,"customer_id","left").show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          5|       Farhan|    Delhi|    NULL|   NULL|  NULL|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
|          4|        Sneha|  Chennai|    NULL|   NULL|  NULL|
+-----------+-------------+---------+--------+-------+------+



In [285]:
customers_df.join(orders_df,"customer_id","right").show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          7|         NULL|     NULL|     105|  Watch|  8000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [286]:
customers_df.join(orders_df,"customer_id","full").show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
|          4|        Sneha|  Chennai|    NULL|   NULL|  NULL|
|          5|       Farhan|    Delhi|    NULL|   NULL|  NULL|
|          7|         NULL|     NULL|     105|  Watch|  8000|
+-----------+-------------+---------+--------+-------+------+



In [287]:
from pyspark.sql.window import Window
from pyspark.sql.functions import  *

In [288]:
employees = [

(101,"Rahul","IT",75000),
(102,"Priya","IT",85000),
(103,"Amit","IT",65000),

(104,"Sneha","HR",70000),
(105,"Farhan","HR",90000),

(106,"Neha","Finance",95000),
(107,"Arjun","Finance",80000),
(108,"Meera","Finance",75000)

]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "salary"
]

df = spark.createDataFrame(
    employees,
    columns
)

df.show()


+-----------+-------------+----------+------+
|employee_id|employee_name|department|salary|
+-----------+-------------+----------+------+
|        101|        Rahul|        IT| 75000|
|        102|        Priya|        IT| 85000|
|        103|         Amit|        IT| 65000|
|        104|        Sneha|        HR| 70000|
|        105|       Farhan|        HR| 90000|
|        106|         Neha|   Finance| 95000|
|        107|        Arjun|   Finance| 80000|
|        108|        Meera|   Finance| 75000|
+-----------+-------------+----------+------+



In [289]:
window_spec=Window.orderBy(col("salary").desc())
df.withColumn("row_number",row_number().over(window_spec)).show()

+-----------+-------------+----------+------+----------+
|employee_id|employee_name|department|salary|row_number|
+-----------+-------------+----------+------+----------+
|        106|         Neha|   Finance| 95000|         1|
|        105|       Farhan|        HR| 90000|         2|
|        102|        Priya|        IT| 85000|         3|
|        107|        Arjun|   Finance| 80000|         4|
|        101|        Rahul|        IT| 75000|         5|
|        108|        Meera|   Finance| 75000|         6|
|        104|        Sneha|        HR| 70000|         7|
|        103|         Amit|        IT| 65000|         8|
+-----------+-------------+----------+------+----------+



In [290]:
window_spec=Window.orderBy(col("salary").desc())
df.withColumn("rank",rank().over(window_spec)).show()

+-----------+-------------+----------+------+----+
|employee_id|employee_name|department|salary|rank|
+-----------+-------------+----------+------+----+
|        106|         Neha|   Finance| 95000|   1|
|        105|       Farhan|        HR| 90000|   2|
|        102|        Priya|        IT| 85000|   3|
|        107|        Arjun|   Finance| 80000|   4|
|        101|        Rahul|        IT| 75000|   5|
|        108|        Meera|   Finance| 75000|   5|
|        104|        Sneha|        HR| 70000|   7|
|        103|         Amit|        IT| 65000|   8|
+-----------+-------------+----------+------+----+



In [291]:
window_spec=Window.orderBy(col("salary").desc())
df.withColumn("rank",dense_rank().over(window_spec)).show()

+-----------+-------------+----------+------+----+
|employee_id|employee_name|department|salary|rank|
+-----------+-------------+----------+------+----+
|        106|         Neha|   Finance| 95000|   1|
|        105|       Farhan|        HR| 90000|   2|
|        102|        Priya|        IT| 85000|   3|
|        107|        Arjun|   Finance| 80000|   4|
|        101|        Rahul|        IT| 75000|   5|
|        108|        Meera|   Finance| 75000|   5|
|        104|        Sneha|        HR| 70000|   6|
|        103|         Amit|        IT| 65000|   7|
+-----------+-------------+----------+------+----+



In [292]:
from google.colab import files
uploaded = files.upload()

Saving patients.csv to patients (4).csv


In [293]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("HospitalAnalytics").getOrCreate()

In [294]:
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)

In [295]:
%%writefile appointments.csv
appointment_id,patient_id,doctor_name,department,appointment_date,consultation_fee,status
5001,101,Dr. Ramesh,Cardiology,2025-01-10,1500,Completed
5002,102,Dr. Suresh,Neurology,2025-01-11,2000,Completed
5003,101,Dr. Anita,Dermatology,2025-01-15,1000,Completed
5004,103,Dr. Ramesh,Cardiology,2025-01-20,1500,Cancelled
5005,104,Dr. Priya,Orthopedics,2025-01-22,2500,Completed
5006,105,Dr. Anita,Dermatology,2025-01-25,1000,Pending
5007,107,Dr. Suresh,Neurology,2025-02-01,2000,Completed
5008,110,Dr. Priya,Orthopedics,2025-02-03,2500,Completed
5009,120,Dr. Ramesh,Cardiology,2025-02-05,1500,Completed
5010,108,Dr. Anita,Dermatology,2025-02-10,,Pending


Overwriting appointments.csv


In [296]:
appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

In [297]:
patients_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [298]:
patients_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)



In [299]:
appointments_df.printSchema()

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- consultation_fee: integer (nullable = true)
 |-- status: string (nullable = true)



In [300]:
patients_df.count()

10

In [301]:
appointments_df.count()

10

In [302]:
patients_df.show(5)

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows


In [303]:
appointments_df.show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|O

In [304]:
patients_df.select("city").distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|    Kochi|
|  Chennai|
|   Mumbai|
|     Pune|
|    Delhi|
|Hyderabad|
|     NULL|
+---------+



In [305]:
appointments_df.select("department").distinct().show()

+-----------+
| department|
+-----------+
|  Neurology|
|Dermatology|
| Cardiology|
|Orthopedics|
+-----------+



In [306]:
patients_df.write.mode("overwrite").parquet("patients_parquet")

In [307]:

patients_parquet_df = spark.read.parquet("patients_parquet")

patients_parquet_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [308]:
print("CSV Count :", patients_df.count())
print("Parquet Count :", patients_parquet_df.count())

CSV Count : 10
Parquet Count : 10


In [309]:
patients_df.filter(col("city") == "Hyderabad").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [310]:
patients_df.filter(col("gender") == "Female").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [311]:
patients_df.filter(col("age") > 40).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [312]:
appointments_df.filter(col("status") == "Completed").show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
|          5009|       120| Dr. Ramesh| Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+-

In [313]:
appointments_df.filter(col("status") == "Pending").show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|Pending|
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [314]:
appointments_df.filter(col("consultation_fee") > 1500).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
+--------------+----------+-----------+-----------+----------------+----------------+---------+



In [315]:
patients_df.filter(col("insurance_status") == "Active").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [316]:
patients_df.filter(col("insurance_status") == "Inactive").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [317]:
patients_df.filter(col("blood_group") == "O+").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [318]:
appointments_df.filter(col("department") == "Cardiology").show()

+--------------+----------+-----------+----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh|Cardiology|      2025-01-10|            1500|Completed|
|          5004|       103| Dr. Ramesh|Cardiology|      2025-01-20|            1500|Cancelled|
|          5009|       120| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+----------+----------------+----------------+---------+



In [319]:
from pyspark.sql.functions import *

In [320]:
patients_df.filter(col("city").isNull()).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+



In [321]:
patients_df.filter(col("blood_group").isNull()).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [322]:
appointments_df.filter(col("consultation_fee").isNull()).show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [323]:
patients_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in patients_df.columns
]).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+



In [324]:
appointments_df.na.fill({"consultation_fee":0}).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|O

In [325]:
appointments_df.na.drop(subset=["consultation_fee"]).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|O

In [326]:
patients_quality=patients_df.withColumn("data_quality_status",when(col("city").isNull()|
                                                  col("blood_group").isNull(),"Incomplete").otherwise("Completed"))
patients_quality.show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|          Completed|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|          Completed|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|          Completed|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|         Incomplete|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|          Completed|
|       108|  Meera Nair|    Kochi| 48|F

In [327]:
patients_quality.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|          Completed|    8|
|         Incomplete|    2|
+-------------------+-----+



In [328]:
patients_df.select(upper(col("patient_name")).alias("patient_name_upper")).show()

+------------------+
|patient_name_upper|
+------------------+
|      RAHUL SHARMA|
|       PRIYA REDDY|
|        AMIT KUMAR|
|       SNEHA PATEL|
|        FARHAN ALI|
|        NEHA SINGH|
|       ARJUN VERMA|
|        MEERA NAIR|
|         KIRAN RAO|
|       NISHA REDDY|
+------------------+



In [329]:
patients_df.select(lower(col("patient_name")).alias("patient_name_lower")).show()

+------------------+
|patient_name_lower|
+------------------+
|      rahul sharma|
|       priya reddy|
|        amit kumar|
|       sneha patel|
|        farhan ali|
|        neha singh|
|       arjun verma|
|        meera nair|
|         kiran rao|
|       nisha reddy|
+------------------+



In [330]:
patients_df.select(col("patient_name"),length(col("patient_name"))).show()

+------------+--------------------+
|patient_name|length(patient_name)|
+------------+--------------------+
|Rahul Sharma|                  12|
| Priya Reddy|                  11|
|  Amit Kumar|                  10|
| Sneha Patel|                  11|
|  Farhan Ali|                  10|
|  Neha Singh|                  10|
| Arjun Verma|                  11|
|  Meera Nair|                  10|
|   Kiran Rao|                   9|
| Nisha Reddy|                  11|
+------------+--------------------+



In [331]:
patients_df.select("patient_name",substring("patient_name",1,3).alias("3_letters")).show()

+------------+---------+
|patient_name|3_letters|
+------------+---------+
|Rahul Sharma|      Rah|
| Priya Reddy|      Pri|
|  Amit Kumar|      Ami|
| Sneha Patel|      Sne|
|  Farhan Ali|      Far|
|  Neha Singh|      Neh|
| Arjun Verma|      Arj|
|  Meera Nair|      Mee|
|   Kiran Rao|      Kir|
| Nisha Reddy|      Nis|
+------------+---------+



In [332]:
patients_df.withColumn(
    "age_group",
    when(col("age") < 30,"Young")
    .when(col("age") < 50,"Adult")
    .otherwise("Senior")
).show()

+----------+------------+---------+---+------+-----------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|age_group|
+----------+------------+---------+---+------+-----------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|    Adult|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|    Young|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|    Adult|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|    Adult|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|   Senior|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|    Adult|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|    Young|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|    Adult|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       N

In [333]:
patients_df.withColumn("insurance_flag",when(col("insurance_status")=="Active",1).otherwise(0)).show()

+----------+------------+---------+---+------+-----------+----------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|insurance_flag|
+----------+------------+---------+---+------+-----------+----------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|             1|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|             1|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|             0|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|             1|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|             1|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|             0|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|             1|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|             1|

In [334]:
patients_df.withColumn(
    "senior_citizen",
    when(col("age") >= 60,"Yes")
    .otherwise("No")
).show()

+----------+------------+---------+---+------+-----------+----------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|            No|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|            No|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|            No|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|            No|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|            No|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|            No|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|            No|

In [335]:
patients_df.select(
    concat_ws("","patient_name","city").alias("patient_city")
).show(truncate=False)

+---------------------+
|patient_city         |
+---------------------+
|Rahul SharmaHyderabad|
|Priya ReddyBangalore |
|Amit KumarMumbai     |
|Sneha PatelChennai   |
|Farhan AliDelhi      |
|Neha Singh           |
|Arjun VermaPune      |
|Meera NairKochi      |
|Kiran RaoHyderabad   |
|Nisha ReddyBangalore |
+---------------------+



In [336]:
patients_df.withColumn(
    "trimmed_name",
    trim(col("patient_name"))
).show()

+----------+------------+---------+---+------+-----------+----------------+------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|trimmed_name|
+----------+------------+---------+---+------+-----------+----------------+------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|Rahul Sharma|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active| Priya Reddy|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|  Amit Kumar|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active| Sneha Patel|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|  Farhan Ali|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|  Neha Singh|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active| Arjun Verma|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|  Meera Nair|
|       109|   Kiran 

In [337]:
patients_df.withColumn(
    "city_upper",
    upper(col("city"))
).show()

+----------+------------+---------+---+------+-----------+----------------+----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|city_upper|
+----------+------------+---------+---+------+-----------+----------------+----------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active| HYDERABAD|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active| BANGALORE|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|    MUMBAI|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|   CHENNAI|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|     DELHI|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|      NULL|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|      PUNE|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|     KOCHI|
|       109|   Kiran Rao|Hyderabad| 33|  Ma

In [338]:
patients_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|     NULL|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+



In [339]:
patients_df.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+



In [340]:
patients_df.groupBy("blood_group").count().show()

+-----------+-----+
|blood_group|count|
+-----------+-----+
|        AB+|    1|
|       NULL|    1|
|         O+|    2|
|         O-|    1|
|         B+|    2|
|         A+|    3|
+-----------+-----+



In [341]:
appointments_df.groupBy("department").count().show()

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [342]:
patients_df.groupBy("city") \
           .agg(avg("age").alias("avg_age")) \
           .show()

+---------+-------+
|     city|avg_age|
+---------+-------+
|Bangalore|   35.0|
|    Kochi|   48.0|
|  Chennai|   31.0|
|     NULL|   38.0|
|   Mumbai|   42.0|
|     Pune|   26.0|
|    Delhi|   55.0|
|Hyderabad|   34.0|
+---------+-------+



In [343]:
patients_df.groupBy("city") \
           .agg(max("age").alias("max_age")) \
           .show()

+---------+-------+
|     city|max_age|
+---------+-------+
|Bangalore|     41|
|    Kochi|     48|
|  Chennai|     31|
|     NULL|     38|
|   Mumbai|     42|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     35|
+---------+-------+



In [344]:
patients_df.groupBy("city") \
           .agg(min("age").alias("min_age")) \
           .show()

+---------+-------+
|     city|min_age|
+---------+-------+
|Bangalore|     29|
|    Kochi|     48|
|  Chennai|     31|
|     NULL|     38|
|   Mumbai|     42|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     33|
+---------+-------+



In [345]:
appointments_df.groupBy("department") \
               .agg(avg("consultation_fee").alias("avg_fee")) \
               .show()

+-----------+-------+
| department|avg_fee|
+-----------+-------+
|  Neurology| 2000.0|
|Dermatology| 1000.0|
| Cardiology| 1500.0|
|Orthopedics| 2500.0|
+-----------+-------+



In [346]:
appointments_df.groupBy("department") \
               .agg(sum("consultation_fee").alias("total_fee")) \
               .show()

+-----------+---------+
| department|total_fee|
+-----------+---------+
|  Neurology|     4000|
|Dermatology|     2000|
| Cardiology|     4500|
|Orthopedics|     5000|
+-----------+---------+



In [347]:
appointments_df.groupBy("department") \
               .agg(sum("consultation_fee").alias("total_revenue")) \
               .orderBy(desc("total_revenue")) \
               .show(1)

+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|Orthopedics|         5000|
+-----------+-------------+
only showing top 1 row


In [348]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

joined_df = patients_df.join(
    appointments_df,
    "patient_id"
)

In [349]:

patients_df.join(
    appointments_df,
    "patient_id",
    "inner"
).show()

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|

In [350]:
patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).show()

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|

In [351]:
patients_df.join(
    appointments_df,
    "patient_id",
    "right"
).show()

+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       103|  Amit Kumar|   Mumbai|  42|

In [352]:
patients_df.join(
    appointments_df,
    "patient_id",
    "full"
).show()

+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       103|  Amit Kumar|   Mumbai|  42|

In [353]:
patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).filter(
    col("appointment_id").isNull()
).show()

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+----------+----------------+----------------+------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name|department|appointment_date|consultation_fee|status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+----------+----------------+----------------+------+
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|          NULL|       NULL|      NULL|            NULL|            NULL|  NULL|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|          NULL|       NULL|      NULL|            NULL|            NULL|  NULL|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+----------+----------------+----------------+------+



In [354]:
appointments_df.join(
    patients_df,
    "patient_id",
    "left"
).filter(
    col("patient_name").isNull()
).show()

+----------+--------------+-----------+----------+----------------+----------------+---------+------------+----+----+------+-----------+----------------+
|patient_id|appointment_id|doctor_name|department|appointment_date|consultation_fee|   status|patient_name|city| age|gender|blood_group|insurance_status|
+----------+--------------+-----------+----------+----------------+----------------+---------+------------+----+----+------+-----------+----------------+
|       120|          5009| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|        NULL|NULL|NULL|  NULL|       NULL|            NULL|
+----------+--------------+-----------+----------+----------------+----------------+---------+------------+----+----+------+-----------+----------------+



In [355]:
appointments_df.groupBy("patient_id").count() .show()

+----------+-----+
|patient_id|count|
+----------+-----+
|       108|    1|
|       101|    2|
|       103|    1|
|       120|    1|
|       107|    1|
|       102|    1|
|       105|    1|
|       110|    1|
|       104|    1|
+----------+-----+



In [356]:
appointments_df.groupBy("patient_id") .agg(
                   sum("consultation_fee")
                   .alias("total_fee")).show()

+----------+---------+
|patient_id|total_fee|
+----------+---------+
|       108|     NULL|
|       101|     2500|
|       103|     1500|
|       120|     1500|
|       107|     2000|
|       102|     2000|
|       105|     1000|
|       110|     2500|
|       104|     2500|
+----------+---------+



In [357]:
appointments_df.groupBy("patient_id") \
               .agg(
                   sum("consultation_fee")
                   .alias("total_fee")
               ) \
               .orderBy(desc("total_fee")) \
               .show(1)

+----------+---------+
|patient_id|total_fee|
+----------+---------+
|       110|     2500|
+----------+---------+
only showing top 1 row


In [358]:
appointments_df.groupBy("patient_id") \
               .agg(
                   count("appointment_id")
                   .alias("appointment_count")
               ) \
               .show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       108|                1|
|       101|                2|
|       103|                1|
|       120|                1|
|       107|                1|
|       102|                1|
|       105|                1|
|       110|                1|
|       104|                1|
+----------+-----------------+



In [359]:
patient_fee_df = appointments_df.groupBy("patient_id") \
    .agg(
        sum("consultation_fee")
        .alias("total_fee")
    )

In [360]:
window_spec = Window.orderBy(desc("total_fee"))

patient_fee_df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+----------+---------+----+
|patient_id|total_fee|rank|
+----------+---------+----+
|       101|     2500|   1|
|       110|     2500|   1|
|       104|     2500|   1|
|       107|     2000|   4|
|       102|     2000|   4|
|       103|     1500|   6|
|       120|     1500|   6|
|       105|     1000|   8|
|       108|     NULL|   9|
+----------+---------+----+



In [361]:
patient_fee_df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
).show()

+----------+---------+----------+
|patient_id|total_fee|dense_rank|
+----------+---------+----------+
|       101|     2500|         1|
|       110|     2500|         1|
|       104|     2500|         1|
|       107|     2000|         2|
|       102|     2000|         2|
|       103|     1500|         3|
|       120|     1500|         3|
|       105|     1000|         4|
|       108|     NULL|         5|
+----------+---------+----------+



In [362]:
patient_fee_df.withColumn(
    "row_num",
    row_number().over(window_spec)
).show()

+----------+---------+-------+
|patient_id|total_fee|row_num|
+----------+---------+-------+
|       101|     2500|      1|
|       110|     2500|      2|
|       104|     2500|      3|
|       107|     2000|      4|
|       102|     2000|      5|
|       103|     1500|      6|
|       120|     1500|      7|
|       105|     1000|      8|
|       108|     NULL|      9|
+----------+---------+-------+



In [363]:
patient_fee_df.withColumn(
    "rank",
    rank().over(window_spec)
).filter(
    col("rank") == 1
).show()

+----------+---------+----+
|patient_id|total_fee|rank|
+----------+---------+----+
|       101|     2500|   1|
|       110|     2500|   1|
|       104|     2500|   1|
+----------+---------+----+



In [364]:
patient_fee_df.withColumn(
    "rank",
    rank().over(window_spec)
).filter(
    col("rank") <= 3
).show()

+----------+---------+----+
|patient_id|total_fee|rank|
+----------+---------+----+
|       101|     2500|   1|
|       110|     2500|   1|
|       104|     2500|   1|
+----------+---------+----+



In [365]:
city_fee_df = patients_df.join(
    appointments_df,
    "patient_id"
).groupBy(
    "patient_id",
    "patient_name",
    "city"
).agg(
    sum("consultation_fee")
    .alias("total_fee")
)

In [366]:
city_window = Window.partitionBy("city") \
                    .orderBy(desc("total_fee"))

city_fee_df.withColumn(
    "rank",
    rank().over(city_window)
).filter(
    col("rank") == 1
).show()

+----------+------------+---------+---------+----+
|patient_id|patient_name|     city|total_fee|rank|
+----------+------------+---------+---------+----+
|       110| Nisha Reddy|Bangalore|     2500|   1|
|       104| Sneha Patel|  Chennai|     2500|   1|
|       105|  Farhan Ali|    Delhi|     1000|   1|
|       101|Rahul Sharma|Hyderabad|     2500|   1|
|       108|  Meera Nair|    Kochi|     NULL|   1|
|       103|  Amit Kumar|   Mumbai|     1500|   1|
|       107| Arjun Verma|     Pune|     2000|   1|
+----------+------------+---------+---------+----+



In [367]:
city_window = Window.partitionBy("city") \
                    .orderBy("total_fee")

city_fee_df.withColumn(
    "rank",
    rank().over(city_window)
).filter(
    col("rank") == 1
).show()

+----------+------------+---------+---------+----+
|patient_id|patient_name|     city|total_fee|rank|
+----------+------------+---------+---------+----+
|       102| Priya Reddy|Bangalore|     2000|   1|
|       104| Sneha Patel|  Chennai|     2500|   1|
|       105|  Farhan Ali|    Delhi|     1000|   1|
|       101|Rahul Sharma|Hyderabad|     2500|   1|
|       108|  Meera Nair|    Kochi|     NULL|   1|
|       103|  Amit Kumar|   Mumbai|     1500|   1|
|       107| Arjun Verma|     Pune|     2000|   1|
+----------+------------+---------+---------+----+



In [368]:
window_spec = Window.orderBy("patient_id")

patient_fee_df.withColumn(
    "running_total",
    sum("total_fee").over(window_spec)
).show()

+----------+---------+-------------+
|patient_id|total_fee|running_total|
+----------+---------+-------------+
|       101|     2500|         2500|
|       102|     2000|         4500|
|       103|     1500|         6000|
|       104|     2500|         8500|
|       105|     1000|         9500|
|       107|     2000|        11500|
|       108|     NULL|        11500|
|       110|     2500|        14000|
|       120|     1500|        15500|
+----------+---------+-------------+



In [369]:
patient_fee_df.withColumn(
    "next_fee",
    lead("total_fee").over(
        Window.orderBy("total_fee")
    )
).show()

+----------+---------+--------+
|patient_id|total_fee|next_fee|
+----------+---------+--------+
|       108|     NULL|    1000|
|       105|     1000|    1500|
|       103|     1500|    1500|
|       120|     1500|    2000|
|       107|     2000|    2000|
|       102|     2000|    2500|
|       101|     2500|    2500|
|       110|     2500|    2500|
|       104|     2500|    NULL|
+----------+---------+--------+



In [370]:
patient_fee_df.withColumn(
    "previous_fee",
    lag("total_fee").over(
        Window.orderBy("total_fee")
    )
).show()

+----------+---------+------------+
|patient_id|total_fee|previous_fee|
+----------+---------+------------+
|       108|     NULL|        NULL|
|       105|     1000|        NULL|
|       103|     1500|        1000|
|       120|     1500|        1500|
|       107|     2000|        1500|
|       102|     2000|        2000|
|       101|     2500|        2000|
|       110|     2500|        2500|
|       104|     2500|        2500|
+----------+---------+------------+



In [371]:
%%writefile patient_preferences.json
 [
  {
    "patient_id": 101,
    "preferred_hospital": "Apollo",
    "contact": {
      "phone": "9876500011",
      "email": "rahul@gmail.com"
    }
  },
  {
    "patient_id": 102,
    "preferred_hospital": "Yashoda",
    "contact": {
      "phone": null,
      "email": "priya@gmail.com"
    }
  },
  {
    "patient_id": 103,
    "preferred_hospital": "Care",
    "contact": {
      "phone": "9876500013",
      "email": null
    }
  },
  {
    "patient_id": 104,
    "preferred_hospital": null,
    "contact": {
      "phone": "9876500014",
      "email": "sneha@gmail.com"
    }
  }
]

Overwriting patient_preferences.json


In [372]:
df = spark.read.option("multiline", True).json("patient_preferences.json")
df.show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [373]:
df.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [374]:
df.select(
    "patient_id",
    col("contact.phone").alias("phone")
).show()

+----------+----------+
|patient_id|     phone|
+----------+----------+
|       101|9876500011|
|       102|      NULL|
|       103|9876500013|
|       104|9876500014|
+----------+----------+



In [375]:
df.select(
    "patient_id",
    col("contact.email").alias("email")
).show()

+----------+---------------+
|patient_id|          email|
+----------+---------------+
|       101|rahul@gmail.com|
|       102|priya@gmail.com|
|       103|           NULL|
|       104|sneha@gmail.com|
+----------+---------------+



In [376]:
df.filter(
    col("contact.phone").isNull()
).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{priya@gmail.com,...|       102|           Yashoda|
+--------------------+----------+------------------+



In [377]:
df.filter(
    col("contact.email").isNull()
).show()

+------------------+----------+------------------+
|           contact|patient_id|preferred_hospital|
+------------------+----------+------------------+
|{NULL, 9876500013}|       103|              Care|
+------------------+----------+------------------+



In [378]:
df.filter(
    col("preferred_hospital").isNull()
).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{sneha@gmail.com,...|       104|              NULL|
+--------------------+----------+------------------+



In [379]:
df = df.withColumn(
    "phone",
    coalesce(col("contact.phone"), lit("Not Available"))
)

df.show()

+--------------------+----------+------------------+-------------+
|             contact|patient_id|preferred_hospital|        phone|
+--------------------+----------+------------------+-------------+
|{rahul@gmail.com,...|       101|            Apollo|   9876500011|
|{priya@gmail.com,...|       102|           Yashoda|Not Available|
|  {NULL, 9876500013}|       103|              Care|   9876500013|
|{sneha@gmail.com,...|       104|              NULL|   9876500014|
+--------------------+----------+------------------+-------------+



In [380]:
df = df.withColumn(
    "email",
    coalesce(col("contact.email"), lit("Not Available"))
)

df.show()

+--------------------+----------+------------------+-------------+---------------+
|             contact|patient_id|preferred_hospital|        phone|          email|
+--------------------+----------+------------------+-------------+---------------+
|{rahul@gmail.com,...|       101|            Apollo|   9876500011|rahul@gmail.com|
|{priya@gmail.com,...|       102|           Yashoda|Not Available|priya@gmail.com|
|  {NULL, 9876500013}|       103|              Care|   9876500013|  Not Available|
|{sneha@gmail.com,...|       104|              NULL|   9876500014|sneha@gmail.com|
+--------------------+----------+------------------+-------------+---------------+



In [381]:
patients_df.join(
    df,
    "patient_id",
    "left"
).show()

+----------+------------+---------+---+------+-----------+----------------+--------------------+------------------+-------------+---------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|             contact|preferred_hospital|        phone|          email|
+----------+------------+---------+---+------+-----------+----------------+--------------------+------------------+-------------+---------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|{rahul@gmail.com,...|            Apollo|   9876500011|rahul@gmail.com|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|{priya@gmail.com,...|           Yashoda|Not Available|priya@gmail.com|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|  {NULL, 9876500013}|              Care|   9876500013|  Not Available|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|{sneha@gmail.com,...|              NULL|   987650

In [382]:
patients_df.createOrReplaceTempView("patients")

In [387]:
appointments_df.createOrReplaceTempView("appointments")

In [383]:
spark.sql("""
SELECT *
FROM patients
""").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [384]:
spark.sql("""
SELECT *
FROM patients
WHERE city='Hyderabad'
""").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [385]:
spark.sql("""
SELECT city,
       COUNT(*) AS total_patients
FROM patients
GROUP BY city
""").show()

+---------+--------------+
|     city|total_patients|
+---------+--------------+
|Bangalore|             2|
|    Kochi|             1|
|  Chennai|             1|
|     NULL|             1|
|   Mumbai|             1|
|     Pune|             1|
|    Delhi|             1|
|Hyderabad|             2|
+---------+--------------+



In [388]:
spark.sql("""
SELECT department,
       COUNT(*) AS total_appointments
FROM appointments
GROUP BY department
""").show()

+-----------+------------------+
| department|total_appointments|
+-----------+------------------+
|  Neurology|                 2|
|Dermatology|                 3|
| Cardiology|                 3|
|Orthopedics|                 2|
+-----------+------------------+



In [389]:
spark.sql("""
SELECT department,
       AVG(consultation_fee) AS avg_fee
FROM appointments
GROUP BY department
""").show()

+-----------+-------+
| department|avg_fee|
+-----------+-------+
|  Neurology| 2000.0|
|Dermatology| 1000.0|
| Cardiology| 1500.0|
|Orthopedics| 2500.0|
+-----------+-------+



In [390]:
spark.sql("""
SELECT department,
       AVG(consultation_fee) AS avg_fee
FROM appointments
GROUP BY department
""").show()

+-----------+-------+
| department|avg_fee|
+-----------+-------+
|  Neurology| 2000.0|
|Dermatology| 1000.0|
| Cardiology| 1500.0|
|Orthopedics| 2500.0|
+-----------+-------+



In [391]:
spark.sql("""
SELECT MAX(consultation_fee) AS highest_fee
FROM appointments
""").show()

+-----------+
|highest_fee|
+-----------+
|       2500|
+-----------+



In [392]:
spark.sql("""
SELECT patient_id,
       COUNT(*) AS appointment_count
FROM appointments
GROUP BY patient_id
""").show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       108|                1|
|       101|                2|
|       103|                1|
|       120|                1|
|       107|                1|
|       102|                1|
|       105|                1|
|       110|                1|
|       104|                1|
+----------+-----------------+



In [393]:
spark.sql("""
SELECT patient_id,
       SUM(consultation_fee) AS total_spending
FROM appointments
GROUP BY patient_id
ORDER BY total_spending DESC
LIMIT 5
""").show()

+----------+--------------+
|patient_id|total_spending|
+----------+--------------+
|       101|          2500|
|       110|          2500|
|       104|          2500|
|       107|          2000|
|       102|          2000|
+----------+--------------+



In [394]:
patients_df = patients_df.na.fill({
    "city":"Unknown",
    "blood_group":"Not Available"
})

appointments_df = appointments_df.na.fill({
    "consultation_fee":0
})

In [395]:
final_df = patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).join(
    df,
    "patient_id",
    "left"
)

In [396]:
patient_spending = final_df.groupBy(
    "patient_id",
    "patient_name"
).agg(
    sum("consultation_fee")
    .alias("total_spending")
)

patient_spending.show()

+----------+------------+--------------+
|patient_id|patient_name|total_spending|
+----------+------------+--------------+
|       107| Arjun Verma|          2000|
|       108|  Meera Nair|             0|
|       109|   Kiran Rao|          NULL|
|       110| Nisha Reddy|          2500|
|       105|  Farhan Ali|          1000|
|       101|Rahul Sharma|          2500|
|       104| Sneha Patel|          2500|
|       103|  Amit Kumar|          1500|
|       106|  Neha Singh|          NULL|
|       102| Priya Reddy|          2000|
+----------+------------+--------------+



In [397]:
department_revenue = final_df.groupBy(
    "department"
).agg(
    sum("consultation_fee")
    .alias("revenue")
)

department_revenue.show()

+-----------+-------+
| department|revenue|
+-----------+-------+
|       NULL|   NULL|
|  Neurology|   4000|
|Dermatology|   2000|
| Cardiology|   3000|
|Orthopedics|   5000|
+-----------+-------+



In [398]:
print("===== HOSPITAL ANALYTICS REPORT =====")

print("Total Patients:",
      patients_df.count())

print("Total Appointments:",
      appointments_df.count())

print("Total Revenue:")

final_df.agg(
    sum("consultation_fee")
).show()

print("Top Spending Patients:")

patient_spending.orderBy(
    desc("total_spending")
).show(5)

print("Department Revenue:")

department_revenue.orderBy(
    desc("revenue")
).show()

===== HOSPITAL ANALYTICS REPORT =====
Total Patients: 10
Total Appointments: 10
Total Revenue:
+---------------------+
|sum(consultation_fee)|
+---------------------+
|                14000|
+---------------------+

Top Spending Patients:
+----------+------------+--------------+
|patient_id|patient_name|total_spending|
+----------+------------+--------------+
|       110| Nisha Reddy|          2500|
|       101|Rahul Sharma|          2500|
|       104| Sneha Patel|          2500|
|       107| Arjun Verma|          2000|
|       102| Priya Reddy|          2000|
+----------+------------+--------------+
only showing top 5 rows
Department Revenue:
+-----------+-------+
| department|revenue|
+-----------+-------+
|Orthopedics|   5000|
|  Neurology|   4000|
| Cardiology|   3000|
|Dermatology|   2000|
|       NULL|   NULL|
+-----------+-------+

